In [4]:
import json
import time
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import random_split
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.preprocessing.sequence import pad_sequences

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/home/qfu1/.local/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
2025-04-01 17:16:37.312044: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-01 17:16:37.321644: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743552997.333575 2927966 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0

In [5]:
with open("/home/qfu1/ML_crystring/ML_crystring/CleanData/token_to_number_combined.json", "r") as f:
    token_to_number_combined = json.load(f)
with open("/home/qfu1/ML_crystring/ML_crystring/CleanData/number_to_token_combined.json", "r") as f:
    number_to_token_combined = json.load(f)

In [6]:
df = pd.read_json("/home/qfu1/ML_crystring/ML_crystring/CleanData/CleanData(Tokenized).json", orient="records")
df

,string,band_gap,energy_above_hull,tokens,tokenized
0,P 4bar n 2 Tb 4f 0.355 0.000 0.000,0.0000,0.000000,"[START, P, 4bar, n, 2, Tb, 4f, 3, 5, 5, 0, 0, ...","[13, 15, 101, 81, 47, 107, 58, 2, 8, 8, 1, 1, ..."
1,P 6bar 1 1 Eu 2g 0.000 0.000 0.545 C 2g 0.000 ...,0.0000,0.000000,"[START, P, 6bar, 1, 1, Eu, 2g, 0, 0, 0, 0, 0, ...","[13, 15, 89, 17, 17, 145, 152, 1, 1, 1, 1, 1, ..."
2,I 4bar c 2 Tl 4a 0.000 0.000 0.000 Cl 8f 0.000...,0.0000,0.000000,"[START, I, 4bar, c, 2, Tl, 4a, 0, 0, 0, 0, 0, ...","[13, 34, 101, 24, 47, 122, 27, 1, 1, 1, 1, 1, ..."
3,P m c 21 Kr 4c 0.595 0.217 0.468 Tb 2a 0.000 0...,0.0000,0.000000,"[START, P, m, c, 21, Kr, 4c, 5, 9, 5, 2, 1, 7,...","[13, 15, 11, 24, 104, 277, 33, 8, 10, 8, 6, 3,..."
4,P 3bar 1 c Co 2c 0.000 0.000 0.000,0.0000,0.000000,"[START, P, 3bar, 1, c, Co, 2c, 0, 0, 0, 0, 0, ...","[13, 15, 20, 17, 24, 72, 52, 1, 1, 1, 1, 1, 1,..."
...,...,...,...,...,...
19084,C m c m Ba 4c 0.000 0.14 0.000 Si 4c 0.000 0.4...,0.0000,0.000000,"[START, C, m, c, m, Ba, 4c, 0, 0, 0, 1, 4, 0, ...","[13, 40, 11, 24, 11, 129, 33, 1, 1, 1, 3, 9, 1..."
19085,C m c m V 8f 0.000 0.296 0.604 O 4c 0.000 0.76...,2.4212,0.032813,"[START, C, m, c, m, V, 8f, 0, 0, 0, 2, 9, 6, 6...","[13, 40, 11, 24, 11, 71, 38, 1, 1, 1, 6, 10, 4..."
19086,C m c m Na 4c 0.000 0.178 0.000 Na 4c 0.000 0....,0.0000,0.000000,"[START, C, m, c, m, Na, 4c, 0, 0, 0, 1, 7, 8, ...","[13, 40, 11, 24, 11, 106, 33, 1, 1, 1, 3, 5, 7..."
19087,C m c m Nb 4c 0.000 0.145 0.000 B 4c 0.000 0.4...,0.0000,0.000000,"[START, C, m, c, m, Nb, 4c, 0, 0, 0, 1, 4, 5, ...","[13, 40, 11, 24, 11, 85, 33, 1, 1, 1, 3, 9, 8,..."


In [7]:
my_input_sequences = []
y = []

for tokens in df['tokenized']:
    for i in range(1, len(tokens)):
        my_input_sequences.append(tokens[:i])
        y.append(tokens[i])

y = np.array(y).reshape(len(y), 1)
max_sequence_len = max([len(seq) for seq in my_input_sequences])
X = np.array(pad_sequences(my_input_sequences, maxlen=max_sequence_len, padding="pre"))

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

# X = torch.nn.functional.one_hot(X, num_classes=vocab_size)
# y = torch.nn.functional.one_hot(y, num_classes=vocab_size)

print(X.shape)
print(y.shape)

torch.Size([1139967, 638])
torch.Size([1139967, 1])


In [8]:
emb_dim = 64
hidden_size = 64
batch_size=512
max_sequence_len = X.shape[1]
vocab_size = len(token_to_number_combined)+1

In [13]:
train_size = int(0.8 * len(X))
test_size = len(X) - train_size
dataset = TensorDataset(X, y)
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [14]:
class VAE_NextToken(nn.Module):  # latent_dim = vocab_size = 249
    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers=3):
        super(VAE_NextToken, self).__init__()
        self.emb_dim = emb_dim
        self.max_sequence_len = max_sequence_len 
        self.vocab_size = vocab_size
        self.num_layers = num_layers
        
        #Encoder
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=emb_dim)
        self.lstm_layer = nn.LSTM(input_size=emb_dim, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.linear_layer_logVar = nn.Linear(in_features=max_sequence_len*hidden_size, out_features=max_sequence_len)
        self.linear_layer_mean = nn.Linear(in_features=max_sequence_len*hidden_size, out_features=max_sequence_len)
        
        #Decoder
        self.Decoder_lstm_layer = nn.LSTM(input_size=emb_dim, hidden_size=vocab_size, num_layers=num_layers, batch_first=True)
        self.conv_layer = nn.Conv1d(in_channels=max_sequence_len, out_channels=1, kernel_size=1)
        self.Decoder_linear_layer = nn.Linear(in_features=max_sequence_len, out_features=max_sequence_len*emb_dim)
        

    def Encoder(self, x):
        # torch.Size([batch_size, 434])
        embedding = self.embedding_layer(x)
        # torch.Size([batch_size, 434, emb_dim])
        rnn_output, (hidden_state, cell_state) = self.lstm_layer(embedding)
        # torch.Size([batch_size, 434, hidden_size])
        rnn_output = rnn_output.reshape(-1, self.max_sequence_len * hidden_size)
        # torch.Size([batch_size, 434*hidden_size])

        mean = self.linear_layer_mean(rnn_output)  
        # torch.Size([batch_size, 434])
        log_var = self.linear_layer_logVar(rnn_output)
        # torch.Size([batch_size, 434])
        return mean, log_var

    def Decoder(self, latent_inputs):
        # torch.Size([batch_size, 434])
        
        linear_output = self.Decoder_linear_layer(latent_inputs)
        linear_output = F.relu(linear_output)
        # torch.Size([batch_size, 434*emb_dim])
        linear_output = linear_output.view(-1, self.max_sequence_len, self.emb_dim)
        # torch.Size([batch_size, 434, emb_dim])
        
        rnn_output, (hidden_state, cell_state) = self.Decoder_lstm_layer(linear_output)
        # rnn_output: torch.Size([batch_size, 434, vocab_size])
        # h0: torch.Size([num_layers, batch_size, vocab_size])
        # c0: torch.Size([num_layers, batch_size, vocab_size])


        final_output = self.conv_layer(rnn_output)
        # torch.Size([batch_size, 1, 249])
        
        return final_output

    def reparameterize(self, mean, log_var):
        # Reparameterization trick
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mean + eps * std
        
    def forward(self, x):
        mean, log_var = self.Encoder(x)
        z = self.reparameterize(mean, log_var)
        recon_x = self.Decoder(z)
        return recon_x, mean, log_var

In [15]:
def vae_loss(recon_x, x, mean, log_var, L=1):
    # x_labels = torch.argmax(x.float(), dim=-1)
    # Reconstruction loss
    recon_loss = nn.CrossEntropyLoss(reduction='sum')(recon_x.view(-1, recon_x.size(-1)), x.view(-1))
    
    # KL Divergence
    kl_loss = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return recon_loss + L*kl_loss

In [16]:
def train_vae(model, train_loader, vae_loss, optimizer, epochs):
    model.to(device)
    model.train()
    losses = []
    start_time = time.time()
    
    for epoch in range(epochs):
        train_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            
            data = data.to(device)
            target = target.to(device)
            
            # Forward pass
            recon_batch, mean, log_var= model(data)
            loss = vae_loss(recon_batch, target, mean, log_var)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            
        avg_loss = train_loss / len(train_loader.dataset)
        losses.append(avg_loss)
        
        if (epoch + 1) % 1 == 0:
            end_time = time.time()
            training_time = end_time - start_time
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}, Time: {training_time:.4f}")
    
    return model, losses

In [17]:
lr=4e-3
wd=1e-5

In [18]:
model = VAE_NextToken(vocab_size, emb_dim, hidden_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
model, losses = train_vae(model, train_loader, vae_loss, optimizer, 10)

Epoch 1/10, Loss: 152.0679, Time: 15.1332
Epoch 2/10, Loss: 3.0618, Time: 30.1975
Epoch 3/10, Loss: 3.0395, Time: 45.3783
Epoch 4/10, Loss: 3.0326, Time: 61.8949
Epoch 5/10, Loss: 3.0256, Time: 92.3139
Epoch 6/10, Loss: 3.0218, Time: 122.9967
Epoch 7/10, Loss: 4.2330, Time: 153.9288
Epoch 8/10, Loss: 3.7265, Time: 184.5091
Epoch 9/10, Loss: 3.0112, Time: 215.2577
Epoch 10/10, Loss: 3.0128, Time: 246.0299


In [50]:
def AutoGenerate(Start: torch.tensor, num_index: int, model: torch.nn.Module, device: torch.device) -> list:
    for _ in range(num_index):
        Start = Start.to(device)
        model.eval()
        mean, log_var = model.Encoder(Start)
        z = model.reparameterize(mean, log_var)
        output = model.Decoder(z)
        # print(Start.shape)
        probs = output.log_softmax(dim=2).flatten().exp()
        index = torch.multinomial(probs, num_samples=1)
        new_input = torch.cat((Start.flatten(), index.flatten()), dim=0).view(1, -1)
        # print(new_input)
        new_input = new_input[new_input.nonzero(as_tuple=True)].cpu().numpy()
        Start = torch.tensor(pad_sequences([new_input], maxlen=max_sequence_len, padding="pre")).to(device)
        # print(Start)
    Pred_index = Start[Start.nonzero(as_tuple=True)].cpu().numpy()
    return Pred_index

In [51]:
def IndexToString(Pred_index: list)-> str:
    predicted_tokens = [number_to_token_combined[index] for index in Pred_index]
    cleaned_tokens = [token.replace('s2_', '').replace('s1_', '') for token in predicted_tokens]
    Pred_str = " ".join(cleaned_tokens)
    return Pred_str

In [52]:
def check_unique_pred_strings(df, model, max_sequence_len, n=20):
    pred_strings = []
    for _ in range(n):
        Start = torch.tensor(pad_sequences([[12]], maxlen=max_sequence_len, padding="pre")).to(device)
        Pred_index = AutoGenerate(Start, 26, model, device)
        Pred_str = IndexToString(Pred_index)
        pred_strings.append(Pred_str)
    return pred_strings

In [53]:
N=20
pred_strings = check_unique_pred_strings(df, model, max_sequence_len, n=N)
pred_strings

['START 8 S 3 0 I 0 Ba 8 0 0 Mg 0 0 3 0 Ce 4b 0 2 3 0 0 5 0 2i 5',
 'START Cs 0 0 9 0 0 0 3 c 0 0 0 S 9 0 0 2 6c 8 0 0 6 6i 0 8 0',
 'START 0 END 12j 0 0 3bar 0 2 END 0 2 5 6 6 0 R 8 3 2 0 2 0 0 7 0 O',
 'START 4e 2 5 1 5 5 4 0 0 3 0 0 0 0 0 9 0 9 7 6 2 6 6j 0 4e 8c',
 'START 2 0 m 7 0 4c 0 4 0 8c 0 0 O 0 8 6bar 2 4 3 3 3 1 4 7 0 2',
 'START 0 0 0 4e 3 7 9 Fe 1 0 1 0 0 4 1 6c O 6 0 0 4c 18f 0 0 3 2i',
 'START 0 7 0 0 0 I 0 2 0 0 4 1 3 0 0 0 Cs F 3 Tl Co 5 0 0 9 6c',
 'START 0 1 I 8g 0 m 2i 1 P 8 0 3 8 0 0 3c 6 1 0 0 0 0 0 5 0 4f',
 'START 0 0 0 0 O 2c 7 9 6 END 3 2 8 0 8 0 0 8f 3 2 Cs 8 8h 0 4 3',
 'START 4 9 4 0 0 0 0 2 S 0 4c 3 2 1 9 7 6 9 0 9 0 0 3 0 0 0',
 'START O 6 8 0 6 4 3 0 0 0 5 0 2i 0 0 4 3 0 0 9 0 0 0 1 P 4',
 'START 3a 8d a 3 1 3 0 1b 9 3 0 3 m 5 Dy 1 8 0 O END I 0 0 9 2a 6',
 'START 8f m 9 4 O 7 m 1 4 0 F 1 0 O 5 0 3 5 4e 2 6 6 m 3 0 0',
 'START 0 0 9 7 O P 0 5 4 0 0 2 0 6c 0 n 0 0 9 0 6 1 0 6 5 0',
 'START 8 0 0 2 0 7 1 8 0 As F 0 m 4 0 2 0 0 0 1 3 3 d 0 3 6',
 'START 3 